**Adding Dependency for the AWS SDK**

In [ ]:
uv add boto3 , botocore


**Importing the Boto3 , BotoCore and JSON**

In [ ]:
import boto3
import botocore
import json

### Configuring the clients 
There are 2 types of client 

1: BedRock
2: Bedrock runtime

In [ ]:
aws_region = "us-east-1"

# Created the Bedrock client
bedrock_client = boto3.client(
    service_name="bedrock",
    region_name=aws_region,
)

# Created the Bedrock runtime client
bedrock_runtime_client = boto3.client(
    service_name="bedrock-runtime",
    region_name=aws_region
    )

# Print the URL for the Bedrock client
print(bedrock_client.meta.endpoint_url)
print(bedrock_runtime_client.meta.endpoint_url)

**Testing the Client Connection**

In [ ]:
print("Testing the Bedrock client")
try:
    response = bedrock_client.list_foundation_models(byOutputModality="TEXT")
    novaModels = [
        model for model in response["modelSummaries"]
        if "nova" in model["modelId"].lower()
    ]
    for model in novaModels:
        print(model)
except Exception as e:
    print(e)

**Creating the Custom GuardRail**

Mentioning Atleast one policy is required to make the guardRail

In [ ]:
bedrock_guardrail = bedrock_client.create_guardrail(
    name="bedrock_guardrail2",
    description="This is a custom guardrail",
    blockedInputMessaging="This input is blocked",
    blockedOutputsMessaging="This output is blocked",
    contentPolicyConfig={
        "filtersConfig": [
            {
                "type": "HATE",
                "inputStrength": "HIGH",
                "outputStrength": "HIGH",
            },
            {
                "type": "VIOLENCE",
                "inputStrength": "MEDIUM",
                "outputStrength": "MEDIUM",
            },
        ]
    },
    topicPolicyConfig={
        "topicsConfig": [
            {
                "name": "medicalAdvice",
                "definition": "This is a medical advice topic",
                "examples": [
                    "I have a headache",
                    "I have a stomach ache",
                    "I have a headache and a stomach ache",
                ],
                "type": "DENY"
            },
        ]
    }
)


print(bedrock_guardrail)


**Deleting the GuardRails**

In [ ]:
response = bedrock_client.delete_guardrail(
    guardrailIdentifier=bedrock_guardrail["guardrailId"]
)
print(response)


**Using Model to get the LLM Response**

Here we are getting results for th medical as we dont have GuardRails Implemeneted

In [ ]:
result = bedrock_runtime_client.invoke_model(
    modelId="amazon.nova-micro-v1:0",
    body=json.dumps({
        "messages": [
            {
                "role": "user",
                "content": [{"text": "I have a headache"}]
            }
        ],
        "inferenceConfig": {
            "maxTokens": 150,
            "temperature": 0.5,
            "topP": 1.0,
        }
    }),
    contentType="application/json",
    accept="application/json",
)

response_body = json.loads(result["body"].read())
print(response_body["output"]["message"]["content"][0]["text"])

**Using Model Along with GuardRails**

In [ ]:
result = bedrock_runtime_client.invoke_model(
    modelId="amazon.nova-micro-v1:0",
    guardrailIdentifier=bedrock_guardrail["guardrailId"],
    guardrailVersion=bedrock_guardrail["version"], 
    body=json.dumps({
        "messages": [
            {
                "role": "user",
                "content": [{"text": "I have a headache"}]
            }
        ],
        "inferenceConfig": {
            "maxTokens": 150,
            "temperature": 0.5,
            "topP": 1.0,
        }
    }),
    
    contentType="application/json",
    accept="application/json",
)

response_body = json.loads(result["body"].read())
print(response_body["output"]["message"]["content"][0]["text"])


In [ ]:
import json

test_texts = [
    {
        "label": "medicalAdvice",
        "text": "I have a headache",
        "source": "INPUT"
    },
    {
        "label": "Cooking Question",
        "text": "What is the recipe for the masala dosa",
        "source": "INPUT"
    }
]

for item in test_texts:

    print(f"Testing the {item['label']} with the {item['source']} text")
    print("=" * 50)

    result = bedrock_runtime_client.apply_guardrail(
        guardrailIdentifier=bedrock_guardrail["guardrailId"],
        guardrailVersion=bedrock_guardrail["version"],
        source=item["source"],
        content=[
            {
                "text": {
                    "text": item["text"]
                }
            }
        ]
    )

    print("Action:", result["action"])

    if "outputs" in result and result["outputs"]:
        print("Output:", result["outputs"][0]["text"])
    else:
        print("No output returned.")

    print()


## Using the Converse Syntax for the GuardRails (Production)

In [ ]:
message = "I need make masala dosa"

result = bedrock_runtime_client.converse(
    # Configuring the Model
    modelId="amazon.nova-pro-v1:0",
    
    # Configuring the Messages
    messages=[
        {
            "role": "user",
            "content": [{"text": message}]
        }
    ],
    
    # Configuring the GuardRails
    guardrailConfig={
        "guardrailIdentifier": bedrock_guardrail["guardrailId"],
        "guardrailVersion": bedrock_guardrail["version"],
        "trace": "ENABLED"
    },
    
    # Configuring the Inference
    inferenceConfig={
        "maxTokens": 150,
        "temperature": 0.5,
        "topP": 1.0,
    }
)
print("=== Full Response ===")
print(result)

print("\n=== Model Response ===")

response_text = result["output"]["message"]["content"][0]["text"]

print(response_text)
